In [ ]:
import polars as pl
from pathlib import Path

In [ ]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD

print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups_map = {
    "plants_plt": ["solanaceae"],
    "plants_mit": ["solanaceae"],
}

In [ ]:
for g, subgs in groups_map.items():
    gdir = BASE / g

    for subg in subgs:
        subg_dir = gdir / subg
        atlas = pl.read_csv(subg_dir / f"{subg}_summary_atlas.tsv", separator="\t")

        gene_pairs = (
            atlas.group_by("gene_pair")
            .agg(
                pl.len().alias("igr_count"),
                pl.col("Length").mean().alias("mean_len"),
                pl.col("Length").median().alias("median_len"),
                pl.col("Length").std().alias("std_len"),
                pl.col("Length").quantile(0.25).alias("q1_len"),
                pl.col("Length").quantile(0.5).alias("q2_len"),
                pl.col("Length").quantile(0.75).alias("q3_len"),
                pl.col("Length").quantile(0.99).alias("q99_len"),
                pl.col("Length").min().alias("min_len"),
                pl.col("Length").max().alias("max_len"),
            )
            .sort("igr_count", descending=True)
        )

        # Round float columns to 2 decimal places
        gene_pairs = gene_pairs.with_columns(pl.col(pl.Float32, pl.Float64).round(2))
        gene_pairs.write_csv(subg_dir / "gene_pairs.tsv", separator="\t")

        flipped = (
            gene_pairs.with_columns(
                pl.col("gene_pair")
                .str.replace(r"_[+-]{2}$", "")
                .alias("gene_pair_cleaned")
            )
            .filter(pl.col("gene_pair_cleaned").count().over("gene_pair_cleaned") > 1)
            .sort("gene_pair")
        )
        flipped.write_csv(subg_dir / "gene_pairs_flipped.tsv", separator="\t")

        filtered = gene_pairs.filter(
            pl.col("igr_count") > 10,
            pl.col("mean_len") > 50,
        ).sort("std_len")
        filtered.write_csv(subg_dir / "gene_pairs_filtered.tsv", separator="\t")

## Extracting fasta sequences

In [ ]:
from collections import defaultdict
from Bio import SeqIO

In [ ]:
source_path = Path("../igannot/sandbox/new_atlas").resolve()
source_path

In [ ]:
for g, subgs in groups_map.items():
    for subg in subgs:
        subg_dir = BASE / g / subg
        print(f"Extracting FASTAs for {g}/{subg}")

        atlas = pl.read_csv(subg_dir / f"{subg}_summary_atlas.tsv", separator="\t")
        filtered_gp = pl.read_csv(subg_dir / "gene_pairs_filtered.tsv", separator="\t")

        # AN -> {ID: gene_pair}, only for the filtered gene pairs
        wanted = atlas.filter(
            pl.col("gene_pair").is_in(filtered_gp["gene_pair"].to_list())
        ).select("AN", pl.col("ID").cast(pl.String), "gene_pair")
        id_to_gp_by_an = {}
        for an, id_, gp in wanted.iter_rows():
            id_to_gp_by_an.setdefault(an, {})[id_] = gp

        # parse each genome FASTA once, dispatching records to their gene pair
        records_by_gp = defaultdict(list)
        missing_fastas = []
        for an, id_to_gp in id_to_gp_by_an.items():
            fasta_file = source_path / g / an / f"{an}_intergenic.fasta"
            if not fasta_file.exists():
                missing_fastas.append(an)
                continue
            for rec in SeqIO.parse(fasta_file, "fasta"):
                gp = id_to_gp.get(rec.id)
                if gp is not None:
                    records_by_gp[gp].append(rec)

        # write one FASTA per gene pair
        out_dir = subg_dir / "gene_pair_fastas"
        out_dir.mkdir(parents=True, exist_ok=True)
        for gp, records in records_by_gp.items():
            SeqIO.write(records, out_dir / f"gp_{gp}.fasta", "fasta")

        # sanity check: expected vs extracted
        n_expected = wanted.height
        n_found = sum(len(r) for r in records_by_gp.values())
        print(
            f"  {len(records_by_gp)} gene pairs written, {n_found}/{n_expected} sequences found"
        )
        if missing_fastas:
            print(
                f"  Missing FASTA for {len(missing_fastas)} ANs: {missing_fastas[:5]}..."
            )